## **Проект:** Анализ рынка недвижимости Санкт-Петербурга и Ленинградской области

### Цель проекта
 Провести комплексный анализ рынка недвижимости для агентства из Петрозаводска, планирующего выход на рынок Санкт-Петербурга и Ленинградской области. На основе данных определить наиболее привлекательные сегменты недвижимости, выявить сезонные закономерности и сформулировать рекомендации для эффективной бизнес-стратегии.

### Описание данных

Схема: `real_estate` — содержит 4 таблицы с информацией о продаже жилой недвижимости в Санкт-Петербурге и Ленинградской области.

**1. Таблица `advertisement`** содержит информацию об объявлениях:
- Всего полей: 4
- Связана с: `flats` (через `id`)

**2. Таблица `flats`** хранит информацию о квартирах:
- Всего полей: 17
- Связана с: `advertisement` (через `id`), `city` (через `city_id`), `type` (через `type_id`)

**3. Таблица `city`**- справочник городов:
- Всего полей: 2
- Связана с: `flats` (через `city_id`)

**4. Таблица `type`** - справочник типов населённых пунктов:
- Всего полей: 2
- Связана с: `flats` (через `type_id`)

<a id="top"></a>
# Содержание проекта

<a href="#section1">1. Знакомство с данными и первичный анализ</a><br>
<a href="#section2">2. Решение ad hoc задач</a><br>
&nbsp;&nbsp;&nbsp;&nbsp;<a href="#section2_1">2.1. Анализ времени активности объявлений</a><br>
&nbsp;&nbsp;&nbsp;&nbsp;<a href="#section2_2">2.2. Исследование сезонности на рынке недвижимости</a><br>
&nbsp;&nbsp;&nbsp;&nbsp;<a href="#section2_3">2.3. Анализ рынка недвижимости Ленинградской области</a><br>
<a href="#section3">3. Общие выводы и рекомендации</a>

Перед началом работы установим необходимые библиотеки и полключимся к базе данных.

In [3]:
pip install psycopg2-binary sqlalchemy pandas

  Using cached psycopg2_binary-2.9.11-cp39-cp39-win_amd64.whl (2.7 MB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

password_encoded = quote_plus("...")
conn_string = f"postgresql://praktikum_student:..."
engine = create_engine(conn_string)
df = pd.read_sql("SELECT 1 as test", engine)
print("Подключение успешно!")
display(df)

Подключение успешно!


,test
0,1


<a id="section1"></a>
#### 1. Знакомство с данными и первичный анализ

<a href="#top">↑ Вернуться к содержанию</a>

Перед решением основных задач необходимо познакомиться с данными, оценить их полноту, качество и основные характеристики. Это поможет корректно интерпретировать результаты дальнейшего анализа.

Определим период представленных данных.

In [3]:
# Определяем минимальную и максимальную даты публикации объявлений
df_period = pd.read_sql("""
SELECT
    MIN(first_day_exposition) AS min_date,
    MAX(first_day_exposition) AS max_date
FROM real_estate.advertisement;
""", engine)

display(df_period)

,min_date,max_date
0,2014-11-27,2019-05-03


**Вывод:** База данных содержит объявления за период с `27 ноября 2014` по `3 мая 2019 года`. Для анализа годовой динамики **будем использовать полные годы:** 2015, 2016, 2017, 2018.

Изучим, какие типы населённых пунктов представлены в данных и как распределены объявления между ними.

In [4]:
# Анализ распределения объявлений по типам населённых пунктов
df_settlement = pd.read_sql("""
SELECT 
    t.type AS settlement_type,
    COUNT(DISTINCT c.city_id) AS cities_count,
    COUNT(a.id) AS advertisements_count
FROM real_estate.type t
LEFT JOIN real_estate.flats f ON t.type_id = f.type_id
LEFT JOIN real_estate.city c ON f.city_id = c.city_id   
LEFT JOIN real_estate.advertisement a ON f.id = a.id
GROUP BY t.type
ORDER BY advertisements_count DESC;
""", engine)

display(df_settlement)

,settlement_type,cities_count,advertisements_count
0,город,43,20008
1,посёлок,113,2092
2,деревня,106,945
3,посёлок городского типа,30,363
4,городской посёлок,13,187
5,село,9,32
6,посёлок при железнодорожной станции,6,15
7,садовое товарищество,4,4
8,коттеджный посёлок,3,3
9,садоводческое некоммерческое товарищество,1,1


**Вывод:**
- Данные содержат информацию о 10 типах населённых пунктов
- Большая часть объявлений (около 85%) приходится на города
- Садоводческое некоммерческое товарищество содержит минимальное количество объявлений

Изучим статистику по времени нахождения объявлений в активном состоянии (дни экспозиции).

In [5]:
# Статистика по времени активности объявлений
df_activity = pd.read_sql("""
SELECT
    MIN(days_exposition) AS min_days_exposition,
    MAX(days_exposition) AS max_days_exposition,
    ROUND(AVG(days_exposition::numeric), 2) AS avg_days_exposition,
    PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY days_exposition) AS median_days
FROM real_estate.advertisement;
""", engine)

display(df_activity)

,min_days_exposition,max_days_exposition,avg_days_exposition,median_days
0,1.0,1580.0,180.75,95.0


**Вывод:** Максимальная длительность публикации объявления составляет более 4 лет. Среднее время активности (180 дней) значительно выше медианы (95 дней), что указывает на наличие объявлений с очень длительным сроком продажи.

Рассчитаем процент объявлений, которые были сняты с публикации (предположительно проданы).

In [6]:
# Доля объявлений, снятых с публикации
df_sold = pd.read_sql("""
SELECT 
    ROUND(COUNT(CASE WHEN days_exposition IS NOT NULL THEN 1 END) * 100.0 / COUNT(*), 2) AS sold_percentage
FROM real_estate.advertisement;
""", engine)

display(df_sold)

,sold_percentage
0,86.55


**Вывод:** 86.55% объявлений были сняты с публикации. Если считать, что снятие объявления означает продажу, то мы имеем хорошую выборку для анализа факторов, влияющих на успешность продажи.

Определим, какая часть объявлений приходится на Санкт-Петербург.

In [7]:
# Доля объявлений в Санкт-Петербурге
df_spb = pd.read_sql("""
SELECT 
    ROUND(COUNT(CASE WHEN c.city = 'Санкт-Петербург' THEN 1 END) * 100.0 / COUNT(*), 2) AS spb_percentage
FROM real_estate.advertisement a
JOIN real_estate.flats f ON a.id = f.id
JOIN real_estate.city c ON f.city_id = c.city_id;
""", engine)
display(df_spb)

,spb_percentage
0,66.47


**Вывод:** 66.47% объявлений приходится на Санкт-Петербург. Соотношение между Санкт-Петербургом и Ленинградской областью составляет примерно 2:1, что отражает более высокую активность рынка в мегаполисе.

Рассчитаем основные статистические показатели для цены квадратного метра.

In [8]:
# Статистика стоимости квадратного метра
df_price = pd.read_sql("""
WITH price_per_m2 AS (
    SELECT a.last_price / NULLIF(f.total_area, 0) AS price_per_square_meter
    FROM real_estate.advertisement a
    JOIN real_estate.flats f ON a.id = f.id
    WHERE f.total_area > 0
)
SELECT 
    ROUND(MIN(price_per_square_meter)::numeric, 2) AS min_price,
    ROUND(MAX(price_per_square_meter)::numeric, 2) AS max_price,
    ROUND(AVG(price_per_square_meter)::numeric, 2) AS avg_price,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY price_per_square_meter)::numeric, 2) AS median_price
FROM price_per_m2;
""", engine)

display(df_price)

,min_price,max_price,avg_price,median_price
0,111.84,1907500.0,99432.25,95000.0


**Вывод:** Наблюдается огромный разброс цен: от 112 руб./м² до 1.9 млн руб./м². Средняя цена (99 432 руб./м²) близка к медиане (95 000 руб./м²), что указывает на относительно симметричное распределение цен в центральной части. Однако экстремально высокие значения требуют дальнейшего анализа на наличие выбросов.

Изучим статистические показатели ключевых количественных характеристик недвижимости для выявления аномальных значений.

In [9]:
# Статистика по основным характеристикам недвижимости
df_stats = pd.read_sql("""
WITH stats AS (
    SELECT
        -- Общая площадь
        MIN(total_area) AS min_total_area,
        MAX(total_area) AS max_total_area,
        ROUND(AVG(total_area::numeric), 2) AS avg_total_area,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_area) AS median_total_area,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY total_area) AS p99_total_area,
        
        -- Количество комнат
        MIN(rooms) AS min_rooms,
        MAX(rooms) AS max_rooms,
        ROUND(AVG(rooms::numeric), 2) AS avg_rooms,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rooms) AS median_rooms,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY rooms) AS p99_rooms,
        
        -- Количество балконов
        MIN(balcony) AS min_balcony,
        MAX(balcony) AS max_balcony,
        ROUND(AVG(balcony::numeric), 2) AS avg_balcony,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY balcony) AS median_balcony,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY balcony) AS p99_balcony,
        
        -- Высота потолков
        MIN(ceiling_height) AS min_ceiling_height,
        MAX(ceiling_height) AS max_ceiling_height,
        ROUND(AVG(ceiling_height::numeric), 2) AS avg_ceiling_height,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ceiling_height) AS median_ceiling_height,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS p99_ceiling_height,
        
        -- Этаж
        MIN(floor) AS min_floor,
        MAX(floor) AS max_floor,
        ROUND(AVG(floor::numeric), 2) AS avg_floor,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY floor) AS median_floor,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY floor) AS p99_floor
    FROM real_estate.flats
)
SELECT * FROM stats;
""", engine)

display(df_stats)

,min_total_area,max_total_area,avg_total_area,median_total_area,p99_total_area,min_rooms,max_rooms,avg_rooms,median_rooms,p99_rooms,...,min_ceiling_height,max_ceiling_height,avg_ceiling_height,median_ceiling_height,p99_ceiling_height,min_floor,max_floor,avg_floor,median_floor,p99_floor
0,12.0,900.0,60.33,52.0,197.556995,0,19,2.07,2.0,5.0,...,1.0,100.0,2.77,2.65,3.8211,1,33,5.89,4.0,23.0


**Выводы:**
- Аномально высокие значения присутствуют практически во всех характеристиках: максимальная общая площадь (820 м²), максимальное количество комнат (19), максимальное количество балконов (17)
- Высота потолков 100 метров явно является ошибкой в данных (99-й перцентиль — всего 3.83 м)
- 99-й перцентиль по общей площади (197.6 м²) значительно ниже максимального значения, что подтверждает наличие выбросов
- Значения этажей до 87 этажа выглядят реалистично для современных жилых комплексов

**Решение для дальнейшего анализа:** При проведении основных задач необходимо фильтровать данные, исключая аномальные значения (выбросы), чтобы результаты были репрезентативными для типичных объектов недвижимости.

Подготовим данные, отфильтровав аномальные значения.

In [ ]:
df_filtered = pd.read_sql(""" --Определяем границы нормальных значений по 99-му перцентилю
WITH limits AS (
    SELECT
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats
),
-- Находим ID объявлений без выбросов
filtered_id AS (
    SELECT id
    FROM real_estate.flats  
    WHERE 
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
)
SELECT COUNT(*) AS filtered_flats_count
FROM real_estate.flats
WHERE id IN (SELECT * FROM filtered_id);
""", engine)

display(df_filtered)

,filtered_flats_count
0,19118


**Результат:** Отфильтровано `19118 записей` (основная масса данных сохранена, аномалии исключены). В дальнейших запросах будем использовать этот фильтр.

<a id="section2"></a>
#### 2. Решение ad hoc задач

<a href="#top">↑ Вернуться к содержанию</a>

<a id="section2_1"></a>
##### 2.1 Анализ времени активности объявлений

**Бизнес-вопросы:**
1. Какие сегменты рынка недвижимости Санкт-Петербурга и Ленинградской области имеют наиболее короткие или длинные сроки активности объявлений?
2. Какие характеристики недвижимости влияют на время активности объявлений?
3. Есть ли различия между Санкт-Петербургом и Ленинградской областью?

**Подход к анализу:** Разделим объявления на сегменты по времени активности и сравним характеристики недвижимости в каждом сегменте для двух регионов.

In [ ]:
df_task1 = pd.read_sql(""" --Фильтрация
WITH limits AS (
    SELECT  
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats     
),
filtered_id AS (
    SELECT id
    FROM real_estate.flats  
    WHERE 
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
),
prepared AS (
    SELECT
        c.city,
        CASE 
            WHEN c.city = 'Санкт-Петербург' THEN 'Санкт-Петербург'
            ELSE 'Ленинградская область'
        END AS region,
        CASE 
            WHEN a.days_exposition BETWEEN 1 AND 30 THEN 'до 1 месяца'
            WHEN a.days_exposition BETWEEN 31 AND 90 THEN '1-3 месяца'
            WHEN a.days_exposition BETWEEN 91 AND 180 THEN '3-6 месяцев'
            WHEN a.days_exposition > 180 THEN 'более 6 месяцев'
            WHEN days_exposition IS NULL THEN 'не продано'
        END AS activity_segment,
        f.total_area,
        f.rooms,
        f.balcony,
        f.floors_total,
        (a.last_price / NULLIF(f.total_area, 0)) AS price_per_sqm
    FROM real_estate.advertisement a
    INNER JOIN real_estate.flats f USING (id)
    INNER JOIN real_estate.city c USING (city_id)
    INNER JOIN real_estate.type t USING (type_id)
    WHERE f.id IN (SELECT id FROM filtered_id)
      AND t.type = 'город'
      AND a.last_price > 0
      AND f.total_area > 0
      AND EXTRACT(YEAR FROM a.first_day_exposition) NOT IN (2014, 2019)
)
SELECT
    region,
    activity_segment,
    COUNT(*) AS объявлений,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY region), 2) AS доля_в_регионе,
    ROUND(AVG(price_per_sqm)::numeric, 0) AS средняя_цена_квм,
    ROUND(AVG(total_area)::numeric, 2) AS средняя_площадь,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rooms::numeric) AS медиана_комнат,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY balcony::numeric) AS медиана_балконов,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY floors_total::numeric) AS медиана_этажность_дома
FROM prepared
GROUP BY region, activity_segment
ORDER BY region DESC, 
         CASE activity_segment
            WHEN 'до 1 месяца' THEN 1
            WHEN '1-3 месяца' THEN 2
            WHEN '3-6 месяцев' THEN 3
            WHEN 'более 6 месяцев' THEN 4
            ELSE 5
         END;
""", engine)

display(df_task1)

,region,activity_segment,объявлений,доля_в_регионе,средняя_цена_квм,средняя_площадь,медиана_комнат,медиана_балконов,медиана_этажность_дома
0,Санкт-Петербург,до 1 месяца,1794,15.99,108920.0,54.66,2.0,1.0,10.0
1,Санкт-Петербург,1-3 месяца,3020,26.92,110874.0,56.58,2.0,1.0,12.0
2,Санкт-Петербург,3-6 месяцев,2244,20.01,111974.0,60.55,2.0,1.0,10.0
3,Санкт-Петербург,более 6 месяцев,3506,31.26,114981.0,65.76,2.0,1.0,9.0
4,Санкт-Петербург,не продано,653,5.82,136108.0,81.38,3.0,1.0,9.0
5,Ленинградская область,до 1 месяца,340,12.02,71908.0,48.75,2.0,1.0,5.0
6,Ленинградская область,1-3 месяца,864,30.55,67424.0,50.85,2.0,1.0,5.0
7,Ленинградская область,3-6 месяцев,553,19.55,69809.0,51.83,2.0,1.0,5.0
8,Ленинградская область,более 6 месяцев,873,30.87,68215.0,55.03,2.0,1.0,5.0
9,Ленинградская область,не продано,198,7.00,72926.0,62.78,2.0,1.0,5.0


**Промежуточный вывод:**
 
**1. Сроки активности оъявлений:**
- в Санкт-Петербурге:
•	Наиболее короткие сроки: сегмент "месяц" (15.99% от общего числа объявлений)
•	Наиболее длинные сроки: сегмент "более полугода" (31.26%)
•	Незакрытые объявления составляют 5.82%
в Ленинградской области:
•	Наиболее короткие сроки: сегмент "месяц" (12.02%
•	Наиболее длинные сроки: сегмент "более полугода" (30.87%)
•	Незакрытые объявления составляют 7%

**2. Общие характеристики недвижимости:**
- в Санкт-Петербурге
•	Быстрее продаются: квартиры площадью 54-56 м² со средней ценой 110 тыс. руб./м²
•	Медленнее продаются: объекты площадью 65-66 м² по цене 115 тыс. руб./м²
•	Незакрытые объявления: крупные объекты (81 м²) с высокой ценой (136 тыс. руб./м²)
- в Ленинградской области:
•	Быстрее продаются: компактные варианты 48-50 м² по цене 71-73 тыс. руб./м²
•	Медленнее продаются: объекты 55 м² по цене 68 тыс. руб./м
•	Незакрытые объявления: объекты среднего размера (63 м²) по цене 73 тыс. руб./м²
**Общие наблюдения:**
Количество комнат (преимущественно 2) и наличие балкона (1) не оказывают значимого влияния на сроки продажи.\

**3. Различия:**
Приоритеты покупателей в двух регионах принципиально отличаются:
- В СПб ключевой фактор - цена (малогабаритные квартиры средней ценовой категории продаются быстрее)
- В ЛенОбл решающую роль играет компактность площади (даже относительно дорогие небольшие объекты находят покупателей быстрее).

<a id="section2_2"></a>
##### 2.2. Исследование сезонности на рынке недвижимости

**Бизнес-вопросы:**
1. В какие месяцы наблюдается наибольшая активность в публикации объявлений?
2. В какие месяцы происходит наибольшее количество снятий объявлений (предположительно продаж)?
3. Как сезонность влияет на цену и характеристики продаваемой недвижимости?

In [ ]:
df_task2 = pd.read_sql(""" --Фильтрация
WITH limits AS (
    SELECT  
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats     
),
filtered_id AS (
    SELECT id
    FROM real_estate.flats 
    WHERE 
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
        AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
        
),
--Подготовка данных с месяцами публикации и снятия
dates_data AS (
    SELECT
        a.id,
        EXTRACT(MONTH FROM a.first_day_exposition) AS pub_month,
        EXTRACT(MONTH FROM (a.first_day_exposition + a.days_exposition * INTERVAL '1 day')) AS close_month,
        a.last_price,
        f.total_area,
        a.days_exposition
    FROM real_estate.advertisement a
    JOIN real_estate.flats f using(id)
    JOIN real_estate.city c using(city_id)
    JOIN real_estate.type t using(type_id)
    WHERE f.id IN (SELECT id FROM filtered_id)
      AND t.type = 'город'
      AND a.last_price > 0
      AND f.total_area > 0
      AND EXTRACT(YEAR FROM a.first_day_exposition) NOT IN (2014, 2019)
),
--Активность публикации
pub_stats AS (
    select
        pub_month,
        TO_CHAR(TO_DATE(pub_month::text, 'MM'), 'Month') AS month_name,
        COUNT(*) AS pub_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pub_share,
        ROUND(AVG(last_price / total_area)::numeric, 2) AS avg_price_sqm,
        ROUND(AVG(total_area)::numeric, 2) AS avg_area,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS pub_rank
    FROM dates_data
    GROUP BY pub_month
),
--Активность снятия 
close_stats AS (
    select
       close_month,
        TO_CHAR(TO_DATE(close_month::text, 'MM'), 'Month') AS month_name,
        COUNT(*) AS close_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS close_share,
        ROUND(AVG(last_price / total_area)::numeric, 2) AS avg_price_sqm,
        ROUND(AVG(total_area)::numeric, 2) AS avg_area,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS close_rank
    FROM dates_data
    WHERE days_exposition IS NOT NULL
    GROUP BY close_month
),
--Объединение месяца 
all_months AS (
    SELECT p.pub_month as month FROM pub_stats p
    UNION
    SELECT c.close_month as month FROM close_stats c
)
--Итоговая таблица 
SELECT
    m.month,
    TO_CHAR(TO_DATE(m.month::text, 'MM'), 'Month') AS месяц,
    COALESCE(p.pub_count, 0) AS публикаций,
    COALESCE(p.pub_share, 0) AS доля_публикаций,
    COALESCE(p.pub_rank, 99) AS ранг_публикаций,
    COALESCE(c.close_count, 0) AS продаж,
    COALESCE(c.close_share, 0) AS доля_продаж,
    COALESCE(c.close_rank, 99) AS ранг_продаж,
    COALESCE(p.avg_price_sqm, 0) AS цена_квм_публикаций,
    COALESCE(p.avg_area, 0) AS площадь_публикаций,
    COALESCE(c.avg_price_sqm, 0) AS цена_квм_продаж,
    COALESCE(c.avg_area, 0) AS площадь_продаж
FROM all_months m
FULL JOIN pub_stats p ON m.month = p.pub_month
FULL JOIN close_stats c ON m.month = c.close_month
ORDER BY m.month;
""", engine)

display(df_task2)

,month,месяц,публикаций,доля_публикаций,ранг_публикаций,продаж,доля_продаж,ранг_продаж,цена_квм_публикаций,площадь_публикаций,цена_квм_продаж,площадь_продаж
0,1.0,January,735,5.23,12,1225,9.28,4,106106.24,59.16,104947.31,57.53
1,2.0,February,1369,9.75,3,1048,7.94,9,103058.51,60.10,103883.72,61.12
2,3.0,March,1119,7.97,8,1071,8.12,8,102429.95,60.00,106832.40,60.37
3,4.0,April,1021,7.27,10,1031,7.81,10,102632.41,60.60,102444.24,59.22
4,5.0,May,891,6.34,11,729,5.53,12,102465.12,59.19,99724.07,57.78
5,6.0,June,1224,8.71,5,771,5.84,11,104802.15,58.37,101863.69,59.82
6,7.0,July,1149,8.18,7,1108,8.40,7,104488.96,60.42,102290.72,58.54
7,8.0,August,1166,8.30,6,1137,8.62,6,107034.70,58.99,100036.51,56.83
8,9.0,September,1341,9.55,4,1238,9.38,3,107563.12,61.04,104070.07,57.49
9,10.0,October,1437,10.23,2,1360,10.31,1,104065.11,59.43,104317.33,58.86


**Промежуточный вывод:**

**1. Активные периоды:**
- Наибольшая активность публикаций наблюдается в ноябре (1569 – приблизительно 11%) и октябре (1437 – приблизительно 10%). Пик снятия объявлений - в октябре (1360 – приблизительно 10%) и ноябре (1301 – приблизительно 9,8%). 
- В октябре-ноябре рынок наиболее сбалансирован — предложение сразу находит спрос. Осенью активны и продавцы, и покупатели (возможно, из-за подготовки к зиме, новогодним праздникам или завершения сделок до конца года).

**2. Влияние сезонности:** 
- Летне-осенний период (июль-ноябрь) – продают более дорогие и просторные квартиры (стоимость 104 -107 тыс. за кв.м., площадь 58-61 кв.м. На этот период приходится примерно 30% годовых сделок.
- Весенний спад (март-май) - примерно 99-102 тыс., площадь кв.м., 57-59 кв.м. Доля сделок составляет примерно 20%.
- В зимние месяцы (декабрь-февраль)– стабильные показатели (примерно 104 тыс. за кв.м., площадь 58 кв.м). Процент сделок составляет примерно 25%.

<a id="section2_3"></a>
##### 2.3 Анализ рынка недвижимости Ленинградской области

**Бизнес-вопросы:**
1. В каких населённых пунктах Ленинградской области наиболее активно публикуют объявления?
2. Где самая высокая доля продаж?
3. Какова средняя цена и площадь в разных городах?
4. Где недвижимость продаётся быстрее, а где — медленнее?

In [14]:
df_task3 = pd.read_sql(""" -- Фильтрация
WITH limits AS (
    SELECT  
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY total_area) AS total_area_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY rooms) AS rooms_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY balcony) AS balcony_limit,
        PERCENTILE_DISC(0.99) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_h,
        PERCENTILE_DISC(0.01) WITHIN GROUP (ORDER BY ceiling_height) AS ceiling_height_limit_l
    FROM real_estate.flats     
),
filtered_id AS (
    SELECT id
    FROM real_estate.flats  
    WHERE 
        total_area < (SELECT total_area_limit FROM limits)
        AND (rooms < (SELECT rooms_limit FROM limits) OR rooms IS NULL)
        AND (balcony < (SELECT balcony_limit FROM limits) OR balcony IS NULL)
        AND ((ceiling_height < (SELECT ceiling_height_limit_h FROM limits)
            AND ceiling_height > (SELECT ceiling_height_limit_l FROM limits)) OR ceiling_height IS NULL)
),
--Выборка для городов ленобл
lenobl_a AS (
    SELECT
        c.city,
        a.id,
        a.days_exposition,
        a.last_price,
        f.total_area,
        f.rooms,
        f.balcony
    FROM real_estate.advertisement a
    JOIN real_estate.flats f using(id)
    JOIN real_estate.city c using(city_id)
    JOIN real_estate.type t using(type_id)
    WHERE 
        c.city != 'Санкт-Петербург' -- Ленинградская область
        AND f.id IN (SELECT id FROM filtered_id)
        AND a.last_price > 0
        AND f.total_area > 0
)
SELECT
    l.city AS город,
    COUNT(*) AS объявлений,
    ROUND(COUNT(days_exposition)::numeric * 100.0 / COUNT(*), 2) as доля_продаж,
    ROUND(AVG(last_price / total_area)::numeric, 2) AS средняя_цена_квм,
    ROUND(AVG(total_area)::numeric, 2) AS средняя_площадь,
    ROUND(AVG(days_exposition)::numeric, 0) AS средние_дни_продажи,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rooms::numeric) AS медиана_комнат,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY balcony::numeric) AS медиана_балконов
FROM lenobl_a l
GROUP BY l.city
ORDER BY объявлений DESC
LIMIT 15;
""", engine)

display(df_task3)

,город,объявлений,доля_продаж,средняя_цена_квм,средняя_площадь,средние_дни_продажи,медиана_комнат,медиана_балконов
0,Мурино,568,93.66,85968.38,43.86,149.0,1.0,1.0
1,Кудрово,463,93.74,95420.47,46.20,161.0,1.0,1.0
2,Шушары,404,92.57,78831.93,53.93,152.0,2.0,1.0
3,Всеволожск,356,85.67,69052.79,55.83,190.0,2.0,1.0
4,Парголово,311,92.60,90272.96,51.34,156.0,1.0,1.0
5,Пушкин,278,83.09,104158.94,59.74,197.0,2.0,1.0
6,Гатчина,228,89.04,69004.74,51.02,188.0,2.0,1.0
7,Колпино,227,92.07,75211.73,52.55,147.0,2.0,1.0
8,Выборг,192,87.50,58669.99,56.76,182.0,2.0,0.0
9,Петергоф,154,88.31,85412.48,51.77,197.0,2.0,1.0


**Промежуточный вывод:**

**1. Наиболее активные населенные пункты по публикациям:**
- Мурино – 568 объявлений
- Кудрово – 463 объявлений
- Шушары – 404 объявлений

**2. Наибольшая доля снятых объявлений (потенциальных продаж) наблюдается в:**
- Кудрово – 93.74%
- Мурино – 93.66%
- Парголово – 92.6%

**3. Анализ вариации цен и площади по населённым пунктам:**

Исследование выявило значительную дифференциацию рынка недвижимости по городам Ленинградской области:

- Прямая зависимость цены от близости к Санкт-Петербургу: чем ближе к городу, тем дороже недвижимость.
- Обратная зависимость между ценой и площадью не прослеживается — в премиальных локациях продаются как компактные, так и просторные квартиры.
- Выявленный ценовой разброс в 1.8 раза подтверждает высокую сегментированность рынка и необходимость дифференцированного подхода при выборе локации для бизнеса.

Примеры:

1. Ценовой сегмент:
- Максимальные цены зафиксированы в Пушкине (104 158 руб./м²) и Сестрорецке (103 848 руб./м²) — эти локации, приближенные к Санкт-Петербургу, формируют премиальный кластер с ценами выше 100 тыс. руб./м².
- Минимальная стоимость отмечена в Выборге (58 699 руб./м²) — разрыв с лидерами составляет 1.8 раза, что объясняется удалённостью от мегаполиса.
2. Площадные характеристики:
- Самое просторное жильё представлено в Сестрорецке (62.45 м²) — вероятно, здесь преобладают квартиры повышенной комфортности.
- Наименьшая средняя площадь среди рассмотренных городов — в Пушкине (59.74 м²) и Выборге (56.76 м²), что близко к среднеобластным значениям.

**4.Скорость продаж:**

Быстрее всего продаётся в:
- Колпино – 147 дней
- Мурино – 149 дней
- Кудрово – 161 день

Медленнее всего продаётся в:
- Сестрорецк – 215 дней
- Петергоф – 197 дней
- Пушкин – 197 дней

<a id="section3"></a>
#### 3. Общие выводы и рекомендации
<a href="#top">↑ Вернуться к содержанию</a>

1.	**Что продавать:**
- В Петербурге лучше всего идут маленькие квартиры (студии и однушки) по средней цене (~110 тыс. за м²).
- В области быстрее продаются маленькие квартиры (48-50 м²), даже если цена чуть выше.

2.	**Когда лучше продавать и покупать:**
- Продавать выгоднее всего с августа по октябрь — в это время цены самые высокие.
- Покупать выгоднее апрель-май — цены падают, можно поймать скидки.
- Зимой рынок спокойный — можно и продавать, и покупать без спешки.

3.	**Где лучше вкладывать деньги:**
- Для быстрых денег: Кудрово и Мурино — квартиры здесь продаются быстро.
- Для новых проектов: Шушары — тут много покупателей.
- Дорогие квартиры (в Пушкине, Сестрорецке) продаются долго — только для тех, кто готов ждать.
